### Sumarization App using Lanchain

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

In [ ]:
from langchain_groq import ChatGroq
from langchain.messages import HumanMessage, SystemMessage


text= r"""
Mojo combines the usability of Python with the performance of C, unlocking unparalleled programmability \
of AI hardware and extensibility of AI models.
Mojo is a new programming language that bridges the gap between research and production \ 
by combining the best of Python syntax with systems programming and metaprogramming.
With Mojo, you can write portable code that’s faster than C and seamlessly inter-op with the Python ecosystem.
When we started Modular, we had no intention of building a new programming language. \
But as we were building our platform with the intent to unify the world’s ML/AI infrastructure, \
we realized that programming across the entire stack was too complicated. Plus, we were writing a \
lot of MLIR by hand and not having a good time.
And although accelerators are important, one of the most prevalent and sometimes overlooked "accelerators" \
is the host CPU. Nowadays, CPUs have lots of tensor-core-like accelerator blocks and other AI acceleration \
units, but they also serve as the “fallback” for operations that specialized accelerators don’t handle, \
such as data loading, pre- and post-processing, and integrations with foreign systems. \
"""


message = [
    SystemMessage(content="You are a helpful assistant that summarizes text."),
    HumanMessage(content=f"Please provide short and concise summary of the following text\nText: {text}")
]


llm = ChatGroq(model="qwen/qwen3-32b", temperature=0)



In [ ]:
summary = llm.invoke(message)
print(summary.content)

## Summarizing Using Prompt Templates

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

template = '''

Write a concise summary  of the following text: 
Text: {text}
translate the summary to {language}
'''


prompt = PromptTemplate(
    input_variables=["text", "language"],
    template=template
)

chain = prompt | llm | output_parser

summary = chain.invoke({"text": text, "language": "Tamil"})
print(summary)

### Summarizing using SuffDocumentChain (file like docs)

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from langchain_classic.chains.summarize import load_summarize_chain


with open("/home/aswanth.cp@knackforge.com/Public/Tutorial/udemy/langchain/files/sj.txt", encoding="utf-8") as file:
    text = file.read()

docs = [Document(page_content=text)]

llm = ChatGroq(model="qwen/qwen3-32b", temperature=0)


template = '''
Write a concise summary  of the following text: 
Text: {text}
'''

prompt = PromptTemplate(
    input_variables=["text"],
    template=template
)

chain = load_summarize_chain(llm, prompt=prompt,chain_type="stuff", verbose=True)

output_summary = chain.invoke(docs)
print(output_summary)

In [ ]:
print(output_summary['output_text'])

### Summarizing Large Documents Using map_reduce

In [ ]:
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains.summarize import load_summarize_chain


with open("/home/aswanth.cp@knackforge.com/Public/Tutorial/udemy/langchain/files/sj.txt", encoding="utf-8") as file:
    text = file.read()

llm = ChatGroq(model="qwen/qwen3-32b", temperature=0)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)

chunks = text_splitter.create_documents([text])


chain = load_summarize_chain(llm, chain_type="map_reduce", verbose=True)

output_summary = chain.invoke(chunks)

print(output_summary)

In [ ]:
print(output_summary['output_text'])

# map_reduce wich Custom Prompts

In [ ]:
from langchain_core.prompts import PromptTemplate


map_prompt = '''
Write a short and concise summary of the following:
Text: `{text}`
CONCISE SUMMARY:
'''

map_prompt_template = PromptTemplate(
    input_variables=["text"],
    template=map_prompt
)

combine_prompt = '''

Write a concise summary of the following text that covers the key points.
Add a title to the summary.
Start your summary with an INTRODUCTION PARAGRAPH that gives an overview of the FOLLOWED by BULLET POINTS if possible AND end the summary
with a CONCLUSION PHRASE.
Text: `{text}`
'''

combine_prompt_template = PromptTemplate(
    template=combine_prompt, input_variables=["text"]
)

summary_chain = load_summarize_chain(
    llm,
    chain_type="map_reduce",
    map_prompt=map_prompt_template,
    combine_prompt=combine_prompt_template,
    verbose=False
)

output = summary_chain.invoke(chunks)
print(output['output_text'])

# Summarizing Using the refine Chain

In [14]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains.summarize import load_summarize_chain
from langchain_core.prompts import PromptTemplate


loader = PyPDFLoader('/home/aswanth.cp@knackforge.com/Public/Tutorial/udemy/langchain/files/attention_is_all_you_need.pdf')

data = loader.load()


# split the data into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=10000, chunk_overlap=50)
chunks = text_splitter.split_documents(data)

llm = ChatGroq(model="qwen/qwen3-32b", temperature=0)

chain = load_summarize_chain(
    llm,
    chain_type="refine",
    verbose=True
)

output = chain.invoke(chunks)



> Entering new RefineDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Write a concise summary of the following:


"Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine 

In [15]:
print(output['output_text'])

<think>
Okay, let's see. The user wants me to refine the existing summary based on the new context provided. The original summary was about the Transformer model from the "Attention Is All You Need" paper. The new context includes an example of attention visualizations, specifically Figure 3 showing how attention heads in the encoder self-attention layer 5 handle long-distance dependencies, like the verb 'making' and its completion in the phrase 'making...more difficult'.

First, I need to check if this new context adds any important information that should be included in the summary. The original summary mentions the Transformer's use of self-attention, its advantages in parallelization, and results on translation tasks. The new context provides a concrete example of how the attention mechanism works in practice, highlighting long-distance dependencies and the role of different attention heads.

The original summary doesn't mention anything about attention visualizations or specific e

# refine with Custom Prompts

In [17]:
prompt_template = """
Write a concise summary of the following extractiong the key information:
Text: {text}
concise summary:
"""
initial_prompt = PromptTemplate(
    input_variables=["text"],
    template=prompt_template
)

refine_prompt_template = """
Your job is to produce a final summary. 
I have provide an existing summary upto a certain point: {existing_answer}.
Please refine the existing summary with some context below.
--------------------------
{text}
--------------------------
start the final summary with an INTRODUCTION PARAGRAPH that gives an overview of the FOLLOWED by BULLET POINTS if possible AND end the summary
with a CONCLUSION PHRASE.
"""

refine_prompt = PromptTemplate(
    input_variables=["existing_answer", "text"],
    template=refine_prompt_template
)



chain = load_summarize_chain(
    llm,
    chain_type="refine",
    question_prompt=initial_prompt,
    refine_prompt=refine_prompt,
    return_intermediate_steps=False,
)

output = chain.invoke(chunks)
print(output['output_text'])


<think>
Okay, let's tackle this. The user wants me to refine the existing summary using the new context provided. The existing summary is about the Transformer model from the "Attention Is All You Need" paper. The new context includes some input examples and a figure description about attention heads in layer 5 dealing with anaphora resolution.

First, I need to check if the new context adds any information that should be included in the summary. The figure mentions two attention heads in layer 5 involved in anaphora resolution, specifically focusing on the word "its." The existing summary already covers attention mechanisms, multi-head attention, and visualizations, but maybe this example adds a concrete instance of how attention works in practice.

The user wants the final summary to start with an introduction, followed by bullet points, and end with a conclusion. The existing summary has an introduction and bullet points. The new context's example about "its" and anaphora resolution